# KNN Regressor

## Import Libraries

In [1]:
# Standard library imports for data handling
import pandas as pd
import numpy as np
import joblib

# Import scikit-learn tools for model selection and evaluation
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# Import categorical encoders for handling make and model features
from category_encoders import BinaryEncoder

## Load Final Dataset

In [2]:
# Define the path for the processed combined dataset
DATA_PATH = '../data/processed/combined_final_dataset.csv'

# Read the CSV file into a pandas dataframe
df = pd.read_csv(DATA_PATH)

# Output the dimensions of the loaded dataset for verification
print(f"Loaded prepared dataset: {df.shape[0]} rows x {df.shape[1]} columns")

# Display the first few records of the dataframe
df.head()

Loaded prepared dataset: 680694 rows x 11 columns


,country,make,model,year,age,mileage,transmission,fuelType,mpg,engineSize,price
0,Uk,Vauxhall,Corsa,2018,2,9876.0,Manual,Petrol,46.13158,1.4,10124.340
1,Uk,Vauxhall,Corsa,2019,1,2500.0,Manual,Petrol,45.21561,1.4,15401.580
2,Uk,Vauxhall,Corsa,2017,3,9625.0,Automatic,Petrol,39.88633,1.4,12553.668
3,Uk,Vauxhall,Corsa,2016,4,25796.0,Manual,Petrol,46.13158,1.4,10914.000
4,Uk,Vauxhall,Corsa,2019,1,3887.0,Manual,Petrol,36.22245,1.4,12840.000


## Data Preprocessing

### Handle year column, and EV mpg and engineSize

In [ ]:
# Initialize random seed and test split ratio for reproducibility
RANDOM_SEED = 42
TEST_SIZE = 0.2

# Identify and drop rows with missing or invalid target values
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]

# Output the new dataset shape after target filtering
print(f"Dataset after filtering: {df.shape}")

# Drop the year column to prevent multicollinearity with the age feature
if 'year' in df.columns:
    df = df.drop(columns=['year'])
    print("Dropped 'year' column")

# Create a binary is_ev flag to identify electric vehicles for special handling
df['is_ev'] = (df['fuelType'] == 'Electric').astype(int)

# Output the frequency and percentage of EV records in the dataset
print(f"EV records: {df['is_ev'].sum()} / {len(df)} ({df['is_ev'].mean()*100:.2f}%)")

# Calculate global medians for MPG and engine size from non-electric vehicles
non_ev_mpg_median = df.loc[df['is_ev'] == 0, 'mpg'].median()
non_ev_engine_median = df.loc[df['is_ev'] == 0, 'engineSize'].median()

# Impute electric vehicle MPG values with the calculated non-EV median
df.loc[df['is_ev'] == 1, 'mpg'] = non_ev_mpg_median

# Impute electric vehicle engine size values with the calculated non-EV median
df.loc[df['is_ev'] == 1, 'engineSize'] = non_ev_engine_median

# Output the imputation values used for the EV records
print(f"Imputed EV mpg with {non_ev_mpg_median:.1f}, engineSize with {non_ev_engine_median:.1f}")

Dataset after filtering: (680694, 11)
Dropped 'year' column
EV records: 0 / 680694 (0.00%)
Imputed EV mpg with 25.5, engineSize with 2.4


### Set train/test split

In [4]:
# Define the target variable for the regression problem
TARGET = 'price'

# Separate the feature matrix (X) from the target vector (y)
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Perform a stratified split to ensure consistent EV representation across sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED,
    stratify=X['is_ev']
)

# Output the number of samples in the training and testing partitions
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")

# Calculate sample weights to prioritize UK market accuracy in the model
train_weights = np.where(X_train['country'] == 'UK', 4.0, 0.5)

# Output confirmation of sample weight assignment
print("\nSample weights computed (UK=4.0, US=0.5).")

Training set: 544555 samples
Test set:      136139 samples

Sample weights computed (UK=4.0, US=0.5).


### Encoding Categorical Features

#### One-Hot Encode Low Cardinality Features

In [5]:
# Define categorical features with low cardinality for one-hot encoding
low_cat_features = ['country', 'transmission', 'fuelType']

# Convert categorical variables into dummy features for the training set
X_train = pd.get_dummies(X_train, columns=low_cat_features, drop_first=True)

# Convert categorical variables into dummy features for the test set
X_test = pd.get_dummies(X_test, columns=low_cat_features, drop_first=True)

# Align the train and test dataframes to ensure matching column structures
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Extract and store the final feature names for the deployment bundle
features = list(X_train.columns)

#### Target Encode High Cardinality Features

In [6]:
# Identify high-cardinality features for target encoding
high_cat_features = ['make', 'model']

# Initialize the ColumnTransformer to handle target encoding for specified features
preprocessor = ColumnTransformer(
    transformers=[
        ('binary_enc', BinaryEncoder(), high_cat_features),
    ],
    remainder='passthrough'
)

# Construct the full modeling pipeline including preprocessing and scaling
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('regressor', KNeighborsRegressor(n_jobs=-1))
])

## Overfit test

In [7]:
# Define a range of K values to test for the bias-variance tradeoff
k_test_values = [3, 5, 10, 20, 50]
comparison_results = []

print("Comparing R² scores for different neighbor values...")

for k in k_test_values:
    # Set the number of neighbors in the pipeline and fit
    pipeline.set_params(regressor__n_neighbors=k)
    pipeline.fit(X_train, y_train)
    
    # Evaluate performance
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    
    comparison_results.append({
        'K': k,
        'Train R2': train_score,
        'Test R2': test_score
    })
    print(f"K={k} | Train R²: {train_score:.4f} | Test R²: {test_score:.4f}")

# Display comparison summary
comparison_df = pd.DataFrame(comparison_results).set_index('K')
print("\n=== Neighbor Comparison Summary ===")
print(comparison_df)

Comparing R² scores for different neighbor values...
K=3 | Train R²: 0.9717 | Test R²: 0.9429
K=5 | Train R²: 0.9633 | Test R²: 0.9436
K=10 | Train R²: 0.9522 | Test R²: 0.9393
K=20 | Train R²: 0.9379 | Test R²: 0.9290
K=50 | Train R²: 0.9066 | Test R²: 0.9025

=== Neighbor Comparison Summary ===
    Train R2   Test R2
K                     
3   0.971716  0.942932
5   0.963338  0.943631
10  0.952172  0.939252
20  0.937928  0.929029
50  0.906555  0.902488


## Hyperparameter Tuning

In [8]:
# Define the search space for Random Forest hyperparameter optimization
param_distributions = {
    'regressor__n_neighbors': [3, 5, 11, 21],
    'regressor__weights': ['uniform', 'distance'],
    'regressor__p': [1, 2]
}

# Initialize RandomizedSearchCV to find the optimal model configuration
model_random = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=2, 
    cv=5,      
    scoring='neg_mean_absolute_error',
    verbose=2, 
    random_state=RANDOM_SEED,
    n_jobs=-1  
)

# Output progress message before starting the search process
print("Starting Hyperparameter Tuning")

# Execute the search using the calculated sample weights for training
model_random.fit(X_train, y_train)

# Extract the best estimator found during the search process
best_pipeline = model_random.best_estimator_

# Output the best hyperparameter combination to the console
print("\n=== Best Hyperparameters Found ===")
print(model_random.best_params_)

Starting Hyperparameter Tuning
Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=uniform; total time=23.7min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=distance; total time=23.7min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=distance; total time=23.8min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=uniform; total time=23.8min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=distance; total time=23.8min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=uniform; total time=23.8min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=uniform; total time=23.8min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=uniform; total time=23.8min
[CV] END regressor__n_neighbors=3, regressor__p=1, regressor__weights=distance; total time=23.8min
[CV] END regressor__n_n

### Model Assessment

#### Check for overfitting

In [9]:
# Generate predictions on both sets
y_train_pred = best_pipeline.predict(X_train)
y_test_pred = best_pipeline.predict(X_test)

# Calculate scores
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
r2_diff = train_r2 - test_r2

# Output the overfitting audit results
print("=== Overfitting Checker ===")
print(f"Training R²: {train_r2:.4f}")
print(f"Testing R²:  {test_r2:.4f}")
print(f"R² Difference: {r2_diff:.4f}")

# Logic check for overfitting
if r2_diff > 0.05:
    print("WARNING: High difference suggests model may be overfitting.")
else:
    print("SUCCESS: Low difference suggests model generalizes well.")

=== Overfitting Checker ===
Training R²: 0.9723
Testing R²:  0.9446
R² Difference: 0.0277
SUCCESS: Low difference suggests model generalizes well.


#### R2, MAE, MAPE and RMSE

In [10]:
# Generate final predictions on the raw test set using the tuned pipeline
y_pred = best_pipeline.predict(X_test)

# Calculate Mean Absolute Error (MAE) in currency units
mae = mean_absolute_error(y_test, y_pred)

# Calculate Mean Absolute Percentage Error (MAPE) for error scale context
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

# Calculate Root Mean Squared Error (RMSE) to penalize larger errors
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Calculate the final R-squared score for the predictive model
r2 = r2_score(y_test, y_pred)

# Output the final performance summary to the console
print("\n=== Tuned Pipeline Performance ===")
print(f"MAE:  ${mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: ${rmse:.2f}")
print(f"R²:   {r2:.4f}")


=== Tuned Pipeline Performance ===
MAE:  $1987.12
MAPE: 9.07%
RMSE: $2923.67
R²:   0.9446


#### Model Export

In [11]:
# Initialize the deployment bundle including the pipeline and metadata
deployment_bundle = {
    "pipeline": best_pipeline,
    "non_ev_mpg_median": non_ev_mpg_median,
    "non_ev_engine_median": non_ev_engine_median,
    "feature_columns": list(X_train.columns) 
}

# Define the final export path for the model pickle file
EXPORT_PATH = '../models/model.pkl'

# Save the deployment bundle to the specified directory using joblib
joblib.dump(deployment_bundle, EXPORT_PATH)

# Output confirmation of the successful model export
print(f"\nSuccess! Deployment bundle saved to: {EXPORT_PATH}")


Success! Deployment bundle saved to: ../models/model.pkl
